In [1]:
import numpy as np 
import pandas as pd
import yfinance as yf

In [2]:
start_date = "2023-01-01"
end_date = "2024-12-31"

end_date_fetch = "2025-01-01"

In [3]:
tickers = {
    "HDFCBANK": "HDFCBANK.NS",
    "ICICIBANK": "ICICIBANK.NS",
    "SBIN": "SBIN.NS",
    "KOTAKBANK": "KOTAKBANK.NS",
    "AXISBANK": "AXISBANK.NS",
    "TCS": "TCS.NS",
    "INFY": "INFY.NS",
    "WIPRO": "WIPRO.NS",
    "HCLTECH": "HCLTECH.NS",
    "TECHM": "TECHM.NS",
    "SUNPHARMA": "SUNPHARMA.NS",
    "DRREDDY": "DRREDDY.NS",
    "CIPLA": "CIPLA.NS",
    "DIVISLAB": "DIVISLAB.NS",
    "APOLLOHOSP": "APOLLOHOSP.NS",
    "NIFTY50": "^NSEI"
}

In [4]:
all_data = []

for label, yahoo_symbol in tickers.items():
    print(f"Fetching {label} ({yahoo_symbol})...")
    
    df = yf.download(
        yahoo_symbol,
        start=start_date,
        end=end_date_fetch,
        interval="1d",
        auto_adjust=False,
        progress=False,
        threads=False
    )
    
    if df.empty:
        print(f" No data found for {label}")
        continue

    # Sometimes yfinance gives multi-index columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    df = df.reset_index()
    df["Ticker"] = label
    df["YahooSymbol"] = yahoo_symbol

    all_data.append(df)

Fetching HDFCBANK (HDFCBANK.NS)...
Fetching ICICIBANK (ICICIBANK.NS)...
Fetching SBIN (SBIN.NS)...
Fetching KOTAKBANK (KOTAKBANK.NS)...
Fetching AXISBANK (AXISBANK.NS)...
Fetching TCS (TCS.NS)...
Fetching INFY (INFY.NS)...
Fetching WIPRO (WIPRO.NS)...
Fetching HCLTECH (HCLTECH.NS)...
Fetching TECHM (TECHM.NS)...
Fetching SUNPHARMA (SUNPHARMA.NS)...
Fetching DRREDDY (DRREDDY.NS)...
Fetching CIPLA (CIPLA.NS)...
Fetching DIVISLAB (DIVISLAB.NS)...
Fetching APOLLOHOSP (APOLLOHOSP.NS)...
Fetching NIFTY50 (^NSEI)...


In [5]:
# Combine all raw data
raw_data = pd.concat(all_data, ignore_index=True)


# Reorder columns
preferred_cols = ["Date", "Ticker", "YahooSymbol", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
existing_cols = [col for col in preferred_cols if col in raw_data.columns]
other_cols = [col for col in raw_data.columns if col not in existing_cols]

raw_data = raw_data[existing_cols + other_cols]

# Basic inspection
print("\nRAW DATA SHAPE:")
print(raw_data.shape)

print("\nCOLUMNS:")
print(raw_data.columns.tolist())

print("\nFIRST 10 ROWS:")
print(raw_data.head(10))

print("\nLAST 10 ROWS:")
print(raw_data.tail(10))

print("\nROWS PER TICKER:")
print(raw_data["Ticker"].value_counts())

print("\nMISSING VALUES COUNT:")
print(raw_data.isna().sum())

# Save raw data
raw_data.to_csv("raw_market_data.csv", index=False)
print("\nSaved file: raw_market_data.csv")


RAW DATA SHAPE:
(7856, 9)

COLUMNS:
['Date', 'Ticker', 'YahooSymbol', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

FIRST 10 ROWS:
        Date    Ticker  YahooSymbol        Open        High         Low  \
0 2023-01-02  HDFCBANK  HDFCBANK.NS  813.500000  819.875000  809.275024   
1 2023-01-03  HDFCBANK  HDFCBANK.NS  811.099976  821.500000  811.099976   
2 2023-01-04  HDFCBANK  HDFCBANK.NS  817.500000  822.875000  803.500000   
3 2023-01-05  HDFCBANK  HDFCBANK.NS  807.500000  809.025024  794.700012   
4 2023-01-06  HDFCBANK  HDFCBANK.NS  801.000000  804.549988  789.099976   
5 2023-01-09  HDFCBANK  HDFCBANK.NS  798.000000  805.775024  795.000000   
6 2023-01-10  HDFCBANK  HDFCBANK.NS  800.000000  800.000000  782.500000   
7 2023-01-11  HDFCBANK  HDFCBANK.NS  783.974976  799.000000  780.000000   
8 2023-01-12  HDFCBANK  HDFCBANK.NS  793.900024  801.950012  792.075012   
9 2023-01-13  HDFCBANK  HDFCBANK.NS  799.375000  804.950012  793.000000   

        Close   Adj Close    Vol

In [6]:
raw_data["Date"] = pd.to_datetime(raw_data["Date"])

print("1. DATE RANGE PER TICKER \n")
date_summary = raw_data.groupby("Ticker").agg(
    start_date=("Date", "min"),
    end_date=("Date", "max"),
    rows=("Date", "count"),
    unique_dates=("Date", "nunique")
).reset_index()

print(date_summary)

print("\n2. DUPLICATE CHECK\n")
exact_duplicates = raw_data.duplicated().sum()
ticker_date_duplicates = raw_data.duplicated(subset=["Ticker", "Date"]).sum()

print("Exact duplicate rows:", exact_duplicates)
print("Duplicate Ticker-Date rows:", ticker_date_duplicates)

print("\n3. BASIC PRICE SANITY CHECK\n")
bad_high_low = raw_data[raw_data["High"] < raw_data["Low"]]
bad_open = raw_data[(raw_data["Open"] < raw_data["Low"]) | (raw_data["Open"] > raw_data["High"])]
bad_close = raw_data[(raw_data["Close"] < raw_data["Low"]) | (raw_data["Close"] > raw_data["High"])]
bad_volume = raw_data[raw_data["Volume"] < 0]

print("Rows where High < Low:", len(bad_high_low))
print("Rows where Open outside [Low, High]:", len(bad_open))
print("Rows where Close outside [Low, High]:", len(bad_close))
print("Rows where Volume < 0:", len(bad_volume))

print("\n4. CHECK TRADING DATE MATCH VS NIFTY50\n")
nifty_dates = set(raw_data[raw_data["Ticker"] == "NIFTY50"]["Date"])

missing_summary = []

for ticker in sorted(raw_data["Ticker"].unique()):
    ticker_dates = set(raw_data[raw_data["Ticker"] == ticker]["Date"])
    missing_vs_nifty = sorted(nifty_dates - ticker_dates)
    
    missing_summary.append({
        "Ticker": ticker,
        "MissingDates_vs_NIFTY50": len(missing_vs_nifty)
    })

missing_summary_df = pd.DataFrame(missing_summary)
print(missing_summary_df)

print("\n5. SAMPLE OF MISSING DATES PER TICKER\n")
for ticker in sorted(raw_data["Ticker"].unique()):
    if ticker == "NIFTY50":
        continue
        
    ticker_dates = set(raw_data[raw_data["Ticker"] == ticker]["Date"])
    missing_vs_nifty = sorted(nifty_dates - ticker_dates)
    
    print(f"{ticker}: {len(missing_vs_nifty)} missing dates")
    if len(missing_vs_nifty) > 0:
        print(" First few missing:", missing_vs_nifty[:10])
    print("-" * 40)

1. DATE RANGE PER TICKER 

        Ticker start_date   end_date  rows  unique_dates
0   APOLLOHOSP 2023-01-02 2024-12-31   491           491
1     AXISBANK 2023-01-02 2024-12-31   491           491
2        CIPLA 2023-01-02 2024-12-31   491           491
3     DIVISLAB 2023-01-02 2024-12-31   491           491
4      DRREDDY 2023-01-02 2024-12-31   491           491
5      HCLTECH 2023-01-02 2024-12-31   491           491
6     HDFCBANK 2023-01-02 2024-12-31   491           491
7    ICICIBANK 2023-01-02 2024-12-31   491           491
8         INFY 2023-01-02 2024-12-31   491           491
9    KOTAKBANK 2023-01-02 2024-12-31   491           491
10     NIFTY50 2023-01-02 2024-12-31   491           491
11        SBIN 2023-01-02 2024-12-31   491           491
12   SUNPHARMA 2023-01-02 2024-12-31   491           491
13         TCS 2023-01-02 2024-12-31   491           491
14       TECHM 2023-01-02 2024-12-31   491           491
15       WIPRO 2023-01-02 2024-12-31   491           491

2. 

In [7]:
# Create cleaned dataset
cleaned_data = raw_data.copy()

# Make sure Date is datetime
cleaned_data["Date"] = pd.to_datetime(cleaned_data["Date"])

# Sort rows properly
cleaned_data = cleaned_data.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# Keep only final columns you want in submission
cleaned_data = cleaned_data[
    ["Date", "Ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
]

# Save cleaned dataset
cleaned_data.to_csv("cleaned_market_data.csv", index=False)
cleaned_data.to_excel("cleaned_market_data.xlsx", index=False)
print("Saved:")
print("- cleaned_market_data.csv")
print("- cleaned_market_data.xlsx")

# Create quality summary table
quality_summary = cleaned_data.groupby("Ticker").agg(
    start_date=("Date", "min"),
    end_date=("Date", "max"),
    rows=("Date", "count"),
    unique_dates=("Date", "nunique")
).reset_index()

quality_summary["missing_values"] = 0
quality_summary["duplicate_ticker_date_rows"] = 0
quality_summary["missing_dates_vs_nifty50"] = 0
quality_summary["flag_more_than_5_consecutive_missing_days"] = "No"

quality_summary.to_csv("data_quality_summary.csv", index=False)

print("\nSaved:")
print("- data_quality_summary.csv")

Saved:
- cleaned_market_data.csv
- cleaned_market_data.xlsx

Saved:
- data_quality_summary.csv


In [8]:
print(cleaned_data.head())
print(cleaned_data.shape)
print(cleaned_data.columns.tolist())

        Date      Ticker         Open         High          Low        Close  \
0 2023-01-02  APOLLOHOSP  4488.000000  4516.700195  4446.000000  4454.350098   
1 2023-01-03  APOLLOHOSP  4467.000000  4508.950195  4425.100098  4490.899902   
2 2023-01-04  APOLLOHOSP  4495.000000  4514.100098  4425.000000  4433.299805   
3 2023-01-05  APOLLOHOSP  4455.500000  4455.500000  4356.000000  4429.049805   
4 2023-01-06  APOLLOHOSP  4442.049805  4474.299805  4369.000000  4387.450195   

     Adj Close  Volume  
0  4411.582520  246577  
1  4447.780762  430096  
2  4390.733887  414977  
3  4386.524902  468816  
4  4345.324707  299640  
(7856, 8)
['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
